# 🌍 Module-1 – Military Data Collection and Preparation
**Unified Military Analytics and Comparison Dashboard**

This notebook implements the **data acquisition pipeline** for the *Unified Military Analytics and Comparison Dashboard* project.

The objective of this milestone is to collect **global military statistics** from publicly available sources and convert them into a structured dataset suitable for analysis and dashboard visualization.

The process implemented in this notebook includes:

- Automated web scraping of military statistics pages  
- Extracting country-level military data  
- Parsing HTML content using BeautifulSoup  
- Structuring the data using Pandas DataFrames  
- Exporting the final dataset as a CSV file  

The dataset produced in this notebook will serve as the **raw data foundation** for the next stages of the project, which include data preprocessing, analysis, and dashboard development.

# 📊 Step 1 – Identify Key Military Metrics

To ensure meaningful analysis, a set of **core military indicators** was identified for extraction.

These metrics capture different dimensions of military capability including:

- **Human Resources** – manpower and personnel strength  
- **Air Power** – aircraft and fighter capabilities  
- **Land Strength** – tanks and armored assets  
- **Naval Strength** – fleet and maritime assets  
- **Economic Strength** – defense spending and financial indicators  

### Required Minimum Metrics

Country  
power_index_rank  
total_manpower  
active_personnel  
reserve_personnel  
total_aircraft  
fighter_aircraft  
naval_assets  
tanks  
defense_budget  
gdp  

These indicators provide a **holistic view of military power across countries**.

# 🧩 Step 2 – Define Raw Data Schema

Before scraping begins, a **raw data schema** is defined to structure the extracted data consistently.

### Dataset Structure

- Each **row represents one country**
- Each **column represents a military metric**
- Data is stored in **raw format without preprocessing**

This schema ensures the dataset can later be used for **data cleaning, transformation, and analytical modeling**.

# 📁 Step 3 – Create Data Storage Folder

To organize the scraped datasets, a **dedicated directory named `data`** is created.

This folder will store **all country-level CSV files** generated during the scraping process.

Creating a structured storage directory helps maintain **clean project organization and easier dataset management**.

In [3]:
import os
os.makedirs("data", exist_ok=True)

# ☁️ Step 4 – Mount Google Drive (Optional for Colab)

When running this notebook in **Google Colab**, Google Drive can be mounted to enable **persistent cloud storage**.

This allows the generated datasets to be **saved safely and accessed later for further analysis or visualization tasks**.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 🌐 Step 5 – Scraping Military Data for a Sample Country

Before automating the scraping process, it is important to **test the extraction logic on a single country page**.

In this step, the Global Firepower webpage for **India** is requested and parsed.

The script performs the following operations:

- Sends an HTTP request to retrieve the webpage  
- Parses the HTML content using **BeautifulSoup**  
- Extracts the **power index ranking and score**  
- Collects all military metrics displayed on the page  
- Converts the extracted information into a **structured dictionary**  
- Saves the results as a **CSV file**

This step verifies that the scraping pipeline works correctly before applying it to multiple countries.

In [6]:


import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from google.colab import files

# -----------------------------
# INDIA URL
# -----------------------------
url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id=india"

# -----------------------------
# FETCH PAGE
# -----------------------------
response = requests.get(url, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "lxml")

data = {}

# -----------------------------
# COUNTRY NAME
# -----------------------------
title = soup.find("h1")
data["country"] = title.get_text(strip=True) if title else "India"

# -----------------------------
# POWER INDEX EXTRACTION
# -----------------------------
overview_text = soup.find("span", class_="textNormal")

if overview_text:
    text = overview_text.get_text(" ", strip=True)

    rank_match = re.search(r"ranked\s+(\d+)", text)
    score_match = re.search(r"score of\s+([\d.]+)", text)

    data["power_index_rank"] = rank_match.group(1) if rank_match else None
    data["power_index_score"] = score_match.group(1) if score_match else None
else:
    data["power_index_rank"] = None
    data["power_index_score"] = None

# -----------------------------
# METRIC EXTRACTION
# -----------------------------
metric_blocks = soup.find_all("div", class_="specsGenContainers")

for block in metric_blocks:
    label = block.find("span", class_="textLarge")
    value = block.find("span", class_="textWhite")

    if label and value:
        key = (
            label.get_text(strip=True)
            .replace(":", "")
            .lower()
            .replace(" ", "_")
        )
        data[key] = value.get_text(strip=True)

# -----------------------------
# DISPLAY OUTPUT
# -----------------------------
print("India Data Extracted:\n")
for k, v in data.items():
    print(f"{k}: {v}")

# -----------------------------
# SAVE CSV (COLAB SAFE)
# -----------------------------
df = pd.DataFrame([data])

file_name = "india_test.csv"
df.to_csv(file_name, index=False)

print(f"\nSaved to {file_name}")

# Download file in Colab
files.download(file_name)

India Data Extracted:

country: 2026India Military Strength
power_index_rank: 4
power_index_score: 0.1346
purchasing_power_parity: 3
foreign_exchange/gold: 5
defense_budget: 5
external_debt: 110
square_land_area: 7
coastline_coverage: 91
shared_borders: 129
waterways_(usable): 10
total_population: 2
available_manpower: 2
fit-for-service: 2
reaching_mil_age_annually: 1
tot_mil._personnel_(est.): 4,958,000
active_personnel: 2
reserve_personnel: 8
paramilitary: 2
air_force_personnel*: 3
army_personnel*: 3
navy_personnel*: 4
yearly_mobilization_potential: 23,652,761
aircraft_total: 4
fighters: 4
attack_types: 4
transports_(fixed-wing): 4
trainers: 8
special-mission: 5
tanker_fleet: 11
helicopters: 4
attack_helicopters: 9
tanks: 5
vehicles: 2
self-propelled_artillery: 35
towed_artillery: 3
mlrs_(rocket_artillery): 16
total_assets: 4
total_tonnage: 5
aircraft_carriers: 3
helicopter_carriers: 145
destroyers: 5
frigates: 3
corvettes: 5
submarines: 7
patrol_vessels: 7
mine_warfare: 145
oil_prod

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 🔁 Step 6 – Automating Data Collection for Multiple Countries

After verifying that the extraction logic works correctly, the same process can be repeated for **multiple countries**.

This is done by modifying the **country_id parameter** in the Global Firepower URL.

By repeating the scraping logic for different country pages, we can build a **comprehensive dataset containing military statistics for multiple nations**.

In [9]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from google.colab import files

# -----------------------------
# USA URL
# -----------------------------
url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id=united-states"

# -----------------------------
# FETCH PAGE
# -----------------------------
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers, timeout=15)

soup = BeautifulSoup(response.text, "lxml")

data = {}

# -----------------------------
# COUNTRY NAME
# -----------------------------
title = soup.find("h1")
data["country"] = title.get_text(strip=True) if title else "United States"

# -----------------------------
# POWER INDEX EXTRACTION
# -----------------------------
overview_text = soup.find("span", class_="textNormal")

if overview_text:
    text = overview_text.get_text(" ", strip=True)

    rank_match = re.search(r"ranked\s+(\d+)", text)
    score_match = re.search(r"score of\s+([\d.]+)", text)

    data["power_index_rank"] = rank_match.group(1) if rank_match else None
    data["power_index_score"] = score_match.group(1) if score_match else None
else:
    data["power_index_rank"] = None
    data["power_index_score"] = None

# -----------------------------
# METRIC EXTRACTION
# -----------------------------
metric_blocks = soup.find_all("div", class_="specsGenContainers")

for block in metric_blocks:
    label = block.find("span", class_="textLarge")
    value = block.find("span", class_="textWhite")

    if label and value:
        key = (
            label.get_text(strip=True)
            .replace(":", "")
            .lower()
            .replace(" ", "_")
        )
        data[key] = value.get_text(strip=True)

# -----------------------------
# DISPLAY OUTPUT
# -----------------------------
print("USA Data Extracted:\n")
for k, v in data.items():
    print(f"{k}: {v}")

# -----------------------------
# SAVE CSV (COLAB SAFE)
# -----------------------------
df = pd.DataFrame([data])

file_name = "usa_test.csv"
df.to_csv(file_name, index=False)

print(f"\nSaved to {file_name}")

files.download(file_name)

USA Data Extracted:

country: 2026Afghanistan Military Strength
power_index_rank: 121
power_index_score: 2.7342
purchasing_power_parity: 101
foreign_exchange/gold: 84
defense_budget: 136
external_debt: 8
square_land_area: 41
coastline_coverage: 999
shared_borders: 109
waterways_(usable): 58
total_population: 36
available_manpower: 43
fit-for-service: 49
reaching_mil_age_annually: 26
tot_mil._personnel_(est.): 165,000
active_personnel: 65
reserve_personnel: 145
paramilitary: 16
air_force_personnel*: 145
army_personnel*: 34
navy_personnel*: 145
yearly_mobilization_potential: 831,916
aircraft_total: 108
fighters: 145
attack_types: 145
transports_(fixed-wing): 49
trainers: 145
special-mission: 145
tanker_fleet: 145
helicopters: 93
attack_helicopters: 145
tanks: 145
vehicles: 96
self-propelled_artillery: 145
towed_artillery: 145
mlrs_(rocket_artillery): 145
oil_production: 145
oil_consumption: 38
oil_deficit: -58,000 bbl
oil_proven_reserves: 145
natural_gas_production: 73
natural_gas_consum

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id=china"

# -----------------------------
# FETCH PAGE
# -----------------------------
response = requests.get(url, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "lxml")

data = {}

# -----------------------------
# COUNTRY NAME
# -----------------------------
title = soup.find("h1")
data["country"] = title.get_text(strip=True)

# -----------------------------
# POWER INDEX EXTRACTION
# -----------------------------
overview_text = soup.find("span", class_="textNormal")

if overview_text:
    text = overview_text.get_text(" ", strip=True)
    rank_match = re.search(r"ranked\s+(\d+)", text)
    score_match = re.search(r"score of\s+([\d.]+)", text)

    data["power_index_rank"] = rank_match.group(1) if rank_match else None
    data["power_index_score"] = score_match.group(1) if score_match else None

# -----------------------------
# METRIC EXTRACTION
# -----------------------------
metric_blocks = soup.find_all("div", class_="specsGenContainers")

for block in metric_blocks:
    label = block.find("span", class_="textLarge")
    value = block.find("span", class_="textWhite")

    if label and value:
        key = (
            label.get_text(strip=True)
            .replace(":", "")
            .lower()
            .replace(" ", "_")
        )
        data[key] = value.get_text(strip=True)

# -----------------------------
# PRINT OUTPUT
# -----------------------------
print("China Data Extracted:\n")
for k, v in data.items():
    print(f"{k}: {v}")


# Save CSV
df = pd.DataFrame([data])     # ← THIS LINE IS REQUIRED
df.to_csv("data/china_test.csv", index=False)
print("Saved to data/china_test.csv")


China Data Extracted:

country: 2026China Military Strength
power_index_rank: 3
power_index_score: 0.0919
purchasing_power_parity: 1
foreign_exchange/gold: 1
defense_budget: 2
external_debt: 121
square_land_area: 4
coastline_coverage: 98
shared_borders: 132
waterways_(usable): 6
total_population: 1
available_manpower: 1
fit-for-service: 1
reaching_mil_age_annually: 2
tot_mil._personnel_(est.): 3,170,000
active_personnel: 1
reserve_personnel: 17
paramilitary: 4
air_force_personnel*: 2
army_personnel*: 2
navy_personnel*: 2
yearly_mobilization_potential: 19,560,509
aircraft_total: 3
fighters: 2
attack_types: 3
transports_(fixed-wing): 3
trainers: 4
special-mission: 4
tanker_fleet: 8
helicopters: 3
attack_helicopters: 3
tanks: 1
vehicles: 3
self-propelled_artillery: 2
towed_artillery: 9
mlrs_(rocket_artillery): 1
total_assets: 1
total_tonnage: 2
aircraft_carriers: 2
helicopter_carriers: 2
destroyers: 2
frigates: 1
corvettes: 2
submarines: 2
patrol_vessels: 6
mine_warfare: 2
oil_production:

In [11]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd


url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id=Indonesia"

# -----------------------------
# FETCH PAGE
# -----------------------------
response = requests.get(url, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "lxml")

data = {}

# -----------------------------
# COUNTRY NAME
# -----------------------------
title = soup.find("h1")
data["country"] = title.get_text(strip=True)

# -----------------------------
# POWER INDEX EXTRACTION
# -----------------------------
overview_text = soup.find("span", class_="textNormal")

if overview_text:
    text = overview_text.get_text(" ", strip=True)
    rank_match = re.search(r"ranked\s+(\d+)", text)
    score_match = re.search(r"score of\s+([\d.]+)", text)

    data["power_index_rank"] = rank_match.group(1) if rank_match else None
    data["power_index_score"] = score_match.group(1) if score_match else None

# -----------------------------
# METRIC EXTRACTION
# -----------------------------
metric_blocks = soup.find_all("div", class_="specsGenContainers")

for block in metric_blocks:
    label = block.find("span", class_="textLarge")
    value = block.find("span", class_="textWhite")

    if label and value:
        key = (
            label.get_text(strip=True)
            .replace(":", "")
            .lower()
            .replace(" ", "_")
        )
        data[key] = value.get_text(strip=True)

# -----------------------------
# PRINT OUTPUT
# -----------------------------
print("Indonesia Data Extracted:\n")
for k, v in data.items():
    print(f"{k}: {v}")


# Save CSV
df = pd.DataFrame([data])     # ← THIS LINE IS REQUIRED
df.to_csv("data/indonesia_test.csv", index=False)
print("Saved to data/indonesia_test.csv")

Indonesia Data Extracted:

country: 2026Indonesia Military Strength
power_index_rank: 13
power_index_score: 0.2582
purchasing_power_parity: 8
foreign_exchange/gold: 22
defense_budget: 33
external_debt: 111
square_land_area: 14
coastline_coverage: 106
shared_borders: 71
waterways_(usable): 8
total_population: 4
available_manpower: 4
fit-for-service: 4
reaching_mil_age_annually: 4
tot_mil._personnel_(est.): 1,055,500
active_personnel: 15
reserve_personnel: 20
paramilitary: 8
air_force_personnel*: 23
army_personnel*: 20
navy_personnel*: 11
yearly_mobilization_potential: 4,726,134
aircraft_total: 24
fighters: 39
attack_types: 21
transports_(fixed-wing): 10
trainers: 22
special-mission: 18
tanker_fleet: 16
helicopters: 22
attack_helicopters: 30
tanks: 42
vehicles: 24
self-propelled_artillery: 25
towed_artillery: 33
mlrs_(rocket_artillery): 46
total_assets: 5
total_tonnage: 11
aircraft_carriers: 145
helicopter_carriers: 145
destroyers: 145
frigates: 9
corvettes: 4
submarines: 13
patrol_vesse

In [12]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd


url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id=Pakistan"

# -----------------------------
# FETCH PAGE
# -----------------------------
response = requests.get(url, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "lxml")

data = {}

# -----------------------------
# COUNTRY NAME
# -----------------------------
title = soup.find("h1")
data["country"] = title.get_text(strip=True)

# -----------------------------
# POWER INDEX EXTRACTION
# -----------------------------
overview_text = soup.find("span", class_="textNormal")

if overview_text:
    text = overview_text.get_text(" ", strip=True)
    rank_match = re.search(r"ranked\s+(\d+)", text)
    score_match = re.search(r"score of\s+([\d.]+)", text)

    data["power_index_rank"] = rank_match.group(1) if rank_match else None
    data["power_index_score"] = score_match.group(1) if score_match else None

# -----------------------------
# METRIC EXTRACTION
# -----------------------------
metric_blocks = soup.find_all("div", class_="specsGenContainers")

for block in metric_blocks:
    label = block.find("span", class_="textLarge")
    value = block.find("span", class_="textWhite")

    if label and value:
        key = (
            label.get_text(strip=True)
            .replace(":", "")
            .lower()
            .replace(" ", "_")
        )
        data[key] = value.get_text(strip=True)

# -----------------------------
# PRINT OUTPUT
# -----------------------------
print("Pakistan Data Extracted:\n")
for k, v in data.items():
    print(f"{k}: {v}")


# Save CSV
df = pd.DataFrame([data])     # ← THIS LINE IS REQUIRED
df.to_csv("data/Pakistan_test.csv", index=False)
print("Saved to data/Pakistan_test.csv")


Pakistan Data Extracted:

country: 2026Pakistan Military Strength
power_index_rank: 14
power_index_score: 0.2626
purchasing_power_parity: 26
foreign_exchange/gold: 66
defense_budget: 38
external_debt: 97
square_land_area: 35
coastline_coverage: 44
shared_borders: 120
waterways_(usable): 145
total_population: 5
available_manpower: 7
fit-for-service: 7
reaching_mil_age_annually: 3
tot_mil._personnel_(est.): 1,710,000
active_personnel: 7
reserve_personnel: 16
paramilitary: 6
air_force_personnel*: 9
army_personnel*: 7
navy_personnel*: 13
yearly_mobilization_potential: 4,734,375
aircraft_total: 7
fighters: 6
attack_types: 7
transports_(fixed-wing): 11
trainers: 2
special-mission: 11
tanker_fleet: 13
helicopters: 9
attack_helicopters: 12
tanks: 7
vehicles: 14
self-propelled_artillery: 9
towed_artillery: 5
mlrs_(rocket_artillery): 10
total_assets: 31
total_tonnage: 28
aircraft_carriers: 145
helicopter_carriers: 145
destroyers: 145
frigates: 10
corvettes: 11
submarines: 11
patrol_vessels: 18
m

# 🧩 Step 7 – Combining All Country Datasets

Once individual country datasets are generated, they must be **merged into a single dataset**.

This step performs the following actions:

- Reads all CSV files stored inside the **data folder**
- Loads them into **Pandas DataFrames**
- Concatenates them into one **combined dataset**

The resulting dataset provides a **unified table containing military statistics for all scraped countries**.

In [15]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import time

base_url = "https://www.globalfirepower.com/"
country_list_url = base_url + "countries-listing.php"

# -----------------------------
# STEP 1: GET ALL COUNTRY LINKS
# -----------------------------
response = requests.get(country_list_url, timeout=15)
response.raise_for_status()
soup = BeautifulSoup(response.text, "lxml")

country_links = []

for link in soup.find_all("a", href=True):
    if "country-military-strength-detail.php?country_id=" in link["href"]:
        full_url = base_url + link["href"]
        if full_url not in country_links:
            country_links.append(full_url)

print(f"Total Countries Found: {len(country_links)}")

# -----------------------------
# STEP 2: SCRAPE EACH COUNTRY
# -----------------------------
all_data = []

for idx, url in enumerate(country_links, 1):
    print(f"Scraping {idx}/{len(country_links)}: {url}")

    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "lxml")

        data = {}

        # COUNTRY NAME
        title = soup.find("h1")
        data["country"] = title.get_text(strip=True) if title else None

        # POWER INDEX
        overview_text = soup.find("span", class_="textNormal")
        if overview_text:
            text = overview_text.get_text(" ", strip=True)
            rank_match = re.search(r"ranked\s+(\d+)", text)
            score_match = re.search(r"score of\s+([\d.]+)", text)

            data["power_index_rank"] = rank_match.group(1) if rank_match else None
            data["power_index_score"] = score_match.group(1) if score_match else None

        # METRICS
        metric_blocks = soup.find_all("div", class_="specsGenContainers")

        for block in metric_blocks:
            label = block.find("span", class_="textLarge")
            value = block.find("span", class_="textWhite")

            if label and value:
                key = (
                    label.get_text(strip=True)
                    .replace(":", "")
                    .lower()
                    .replace(" ", "_")
                )
                data[key] = value.get_text(strip=True)

        all_data.append(data)

        time.sleep(1)  # polite delay

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        continue

# -----------------------------
# STEP 3: SAVE ALL DATA
# -----------------------------
df = pd.DataFrame(all_data)
df.to_csv("Military_raw_data.csv", index=False)

print("\nScraping Completed Successfully ✅")
print(f"Total Countries Scraped: {len(df)}")
print("Saved to Military_raw_data.csv")

from google.colab import files
files.download("Military_raw_data.csv")



Total Countries Found: 145
Scraping 1/145: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=united-states-of-america
Scraping 2/145: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=russia
Scraping 3/145: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=china
Scraping 4/145: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=india
Scraping 5/145: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=south-korea
Scraping 6/145: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=france
Scraping 7/145: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=japan
Scraping 8/145: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=united-kingdom
Scraping 9/145: https://www.globalfirepower.com//country-military-strength-detail.php?country_id=turkey
Scraping 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The combined raw military dataset was successfully saved and is ready for data cleaning.